In [1]:
import numpy as np
import versioned_hdf5
import h5py
import pyinstrument


data = np.random.random((40_000, 80 * 1024 // 8))
file = h5py.File("bench.h5", "w")
vf = versioned_hdf5.VersionedHDF5File(file)
with vf.stage_version("r0") as v:
    v.create_dataset("x", data=data, chunks=(1, data.shape[1]))
file.flush()

In [2]:
!ls -lh bench.h5

-rw-rw-r-- 1 crusaderky crusaderky 3.1G May  5 16:56 bench.h5


In [3]:
%%time
with vf.stage_version("update-all-same-hash") as v:
    ds = v["x"]
    ds[:] = data
file.flush()

CPU times: user 4.1 s, sys: 551 ms, total: 4.65 s
Wall time: 4.65 s


In [4]:
data += 1

In [5]:
with pyinstrument.profile():
    with vf.stage_version("update-all") as v:
        ds = v["x"]
        ds[:] = data
    file.flush()


pyinstrument ........................................
.
.  Block at /tmp/ipykernel_777997/234481162.py:1
.
.  14.237 <module>  /tmp/ipykernel_777997/234481162.py:1
.  ├─ 12.966 _GeneratorContextManager.__exit__  contextlib.py:145
.  │  └─ 12.966 VersionedHDF5File.stage_version  versioned_hdf5/api.py:277
.  │     └─ 12.958 commit_version  versioned_hdf5/versions.py:79
.  │        ├─ 11.013 write_dataset  versioned_hdf5/backend.py:250
.  │        │  ├─ 7.050 Dataset.__setitem__  h5py/_hl/dataset.py:962
.  │        │  │     [7 frames hidden]  h5py
.  │        │  │        4.694 [self]  h5py/_hl/dataset.py
.  │        │  ├─ 1.852 Hashtable.hash  versioned_hdf5/hashtable.py:119
.  │        │  │  └─ 1.710 HASH.update  <built-in>
.  │        │  ├─ 0.627 [self]  versioned_hdf5/backend.py
.  │        │  ├─ 0.604 NDIndexConstructor.__call__  ndindex/ndindex.py:112
.  │        │  │  └─ 0.580 NDIndexConstructor.__getitem__  ndindex/ndindex.py:52
.  │        │  ├─ 0.383 Hashtable.__setitem__  versi

In [6]:
with pyinstrument.profile():
    with vf.stage_version("update-one") as v:
        ds = v["x"]
        ds[123, 0] = 1337
    file.flush()


pyinstrument ........................................
.
.  Block at /tmp/ipykernel_777997/770983063.py:1
.
.  4.134 <module>  /tmp/ipykernel_777997/770983063.py:1
.  ├─ 3.369 _GeneratorContextManager.__exit__  contextlib.py:145
.  │  └─ 3.369 VersionedHDF5File.stage_version  versioned_hdf5/api.py:277
.  │     └─ 3.358 commit_version  versioned_hdf5/versions.py:79
.  │        ├─ 2.139 create_virtual_dataset  versioned_hdf5/backend.py:557
.  │        │  ├─ 0.561 [self]  versioned_hdf5/backend.py
.  │        │  ├─ 0.559 select  h5py/_hl/selections.py:19
.  │        │  │     [4 frames hidden]  h5py, weakref
.  │        │  ├─ 0.476 Group.create_virtual_dataset  h5py/_hl/group.py:245
.  │        │  │  └─ 0.476 VirtualLayout.make_dataset  h5py/_hl/vds.py:229
.  │        │  ├─ 0.147 Tuple.isempty  ndindex/tuple.py:672
.  │        │  │     [3 frames hidden]  ndindex
.  │        │  ├─ 0.087 SimpleSelection.__init__  h5py/_hl/selections.py:228
.  │        │  │  └─ 0.064 SimpleSelection.__init__ 